# 🎓 Student Performance Prediction – EDA & Modeling

This notebook walks through the complete ML pipeline:
1. Data Understanding
2. Exploratory Data Analysis (EDA)
3. Preprocessing (no data leakage)
4. Two Experiments (with / without period grades)
5. Model Training & Evaluation
6. Diagnostics
7. Explainability
8. At-Risk Classification

**Dataset**: UCI Student Performance (student-mat.csv, 395 students, 33 features)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.figsize': (10, 6), 'savefig.dpi': 150})

# Load data
df = pd.read_csv('../data/student-mat.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()

## Step 1 – Data Understanding

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData types:\n{df.dtypes.value_counts()}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
df.describe().round(2)

In [ ]:
# G3 distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['G3'], bins=21, kde=True, color='#4C72B0', edgecolor='white', ax=ax)
ax.set_title('Distribution of Final Grade (G3)', fontsize=14, fontweight='bold')
zero_count = (df['G3'] == 0).sum()
ax.axvline(x=0, color='red', linestyle='--', alpha=0.7)
ax.text(0.5, ax.get_ylim()[1]*0.9, f'G3=0: {zero_count} students', color='red')
plt.savefig('../reports/figures/g3_distribution.png', bbox_inches='tight')
plt.show()
print(f'Students with G3=0: {zero_count} ({zero_count/len(df)*100:.1f}%)')

In [ ]:
# At-risk class balance
at_risk = (df['G3'] < 10).astype(int)
counts = at_risk.value_counts()
print(f'Not at risk: {counts[0]}')
print(f'At risk (G3<10): {counts[1]}')
print(f'Ratio: {counts[1]/len(df)*100:.1f}% at risk')

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', color=['#55A868', '#C44E52'], ax=ax)
ax.set_xticklabels(['Not At Risk', 'At Risk'], rotation=0)
ax.set_title('At-Risk Class Balance', fontweight='bold')
plt.savefig('../reports/figures/at_risk_balance.png', bbox_inches='tight')
plt.show()

## Step 2 – EDA

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
fig, ax = plt.subplots(figsize=(14, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (Numeric Features)', fontweight='bold')
plt.savefig('../reports/figures/correlation_heatmap.png', bbox_inches='tight')
plt.show()

**Insight**: G1 and G2 are very strongly correlated with G3 (>0.80). They are intermediate grades — nearly direct proxies for the final grade.

In [ ]:
# Boxplots: G3 vs key features
box_features = ['failures', 'studytime', 'Medu', 'Fedu', 'higher', 'schoolsup']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, feat in enumerate(box_features):
    ax = axes[i//3, i%3]
    sns.boxplot(data=df, x=feat, y='G3', palette='Set2', ax=ax)
    ax.set_title(f'G3 vs {feat}', fontweight='bold')
plt.suptitle('G3 vs Key Features', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots_g3_vs_features.png', bbox_inches='tight')
plt.show()

**Insight**: More past failures → lower G3. Students who want higher education perform significantly better. Extra school support (schoolsup) is given to struggling students, so it correlates with *lower* grades.

In [ ]:
# Scatter: G1 vs G3 and G2 vs G3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, grade in enumerate(['G1', 'G2']):
    ax = axes[i]
    ax.scatter(df[grade], df['G3'], alpha=0.5, s=50, color='#4C72B0')
    z = np.polyfit(df[grade], df['G3'], 1)
    x_line = np.linspace(df[grade].min(), df[grade].max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), '--', color='#C44E52', linewidth=2)
    r = df[grade].corr(df['G3'])
    ax.set_title(f'{grade} vs G3 (r = {r:.3f})', fontweight='bold')
    ax.set_xlabel(grade); ax.set_ylabel('G3')
plt.tight_layout()
plt.savefig('../reports/figures/scatter_g1_g2_vs_g3.png', bbox_inches='tight')
plt.show()

**Insight**: G1 and G2 are almost linearly related to G3. This is why they dominate any predictive model — but also why using them for early prediction is cheating.

## Steps 3-4 – Preprocessing & Experiments

### No Data Leakage Strategy
- Split 80/20 BEFORE any fitting
- Use `Pipeline(ColumnTransformer + Model)` so scalers/encoders fit only on training data

### Two Experiments
- **A**: All features (including G1, G2) — maximum accuracy
- **B**: Early-warning — G1 and G2 dropped

**Why drop G1/G2?** These period grades are recorded during the school year. Using them to predict G3 is near-circular. For a genuine early-warning system, we need to predict BEFORE any grades exist.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.preprocess import load_data, build_pipeline, get_feature_names
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

X, y = load_data('../data/student-mat.csv')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Step 5 – Model Training & Evaluation

Train 6 models per experiment with 5-fold CV and GridSearch for tunable models.

In [ ]:
RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models_config = {
    'Linear Regression': (LinearRegression(), None),
    'Ridge': (Ridge(random_state=RANDOM_STATE), None),
    'Decision Tree': (DecisionTreeRegressor(random_state=RANDOM_STATE),
                      {'model__max_depth': [3, 5, 8], 'model__min_samples_leaf': [2, 5]}),
    'Random Forest': (RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
                      {'model__n_estimators': [100, 200], 'model__max_depth': [5, 10]}),
    'Gradient Boosting': (GradientBoostingRegressor(random_state=RANDOM_STATE),
                          {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1]}),
    'SVR': (SVR(), {'model__C': [0.1, 1, 10], 'model__epsilon': [0.1, 0.5]}),
}

def train_experiment(X_tr, X_te, y_tr, y_te, include_grades):
    results = []
    best_score, best_pipe = -np.inf, None
    for name, (est, grid) in models_config.items():
        pipe = build_pipeline(est, include_grades=include_grades)
        cv = cross_val_score(pipe, X_tr, y_tr, cv=kf, scoring='r2')
        if grid:
            gs = GridSearchCV(pipe, grid, scoring='r2', cv=kf, n_jobs=-1, refit=True)
            gs.fit(X_tr, y_tr)
            pipe = gs.best_estimator_
        else:
            pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        r2 = r2_score(y_te, y_pred)
        mae = mean_absolute_error(y_te, y_pred)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        results.append({'Model': name, 'CV R2': f'{cv.mean():.4f}±{cv.std():.4f}',
                        'Test R2': round(r2, 4), 'MAE': round(mae, 4), 'RMSE': round(rmse, 4)})
        if r2 > best_score:
            best_score, best_pipe = r2, pipe
    return pd.DataFrame(results).sort_values('Test R2', ascending=False), best_pipe

print('=== Experiment A: With G1, G2 ===')
df_a, best_a = train_experiment(X_train, X_test, y_train, y_test, True)
df_a

In [ ]:
print('=== Experiment B: Early Warning (no G1, G2) ===')
df_b, best_b = train_experiment(X_train, X_test, y_train, y_test, False)
df_b

## Step 6 – Diagnostics

In [ ]:
# Actual vs Predicted for best models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (pipe, label) in enumerate([(best_a, 'Exp A'), (best_b, 'Exp B')]):
    ax = axes[i]
    y_pred = pipe.predict(X_test)
    ax.scatter(y_test, y_pred, alpha=0.6, s=50, color='#4C72B0')
    ax.plot([0, 20], [0, 20], '--', color='#C44E52', linewidth=2)
    ax.set_xlabel('Actual G3'); ax.set_ylabel('Predicted G3')
    ax.set_title(f'Actual vs Predicted ({label})', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7 – Feature Importance & Permutation Importance

See `train.py` output and `reports/figures/` for full feature importance plots.

## Step 8 – At-Risk Classification

Convert regression predictions to at-risk (G3 < 10). Recall is the most important metric — missing an at-risk student is the costly error.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

for pipe, label in [(best_a, 'Exp A'), (best_b, 'Exp B')]:
    y_pred = pipe.predict(X_test)
    y_pred_class = (y_pred < 10).astype(int)
    y_true_class = (y_test < 10).astype(int)
    print(f'\n=== {label} ===')
    print(classification_report(y_true_class, y_pred_class, target_names=['Not At Risk', 'At Risk']))
    print('Confusion Matrix:')
    print(confusion_matrix(y_true_class, y_pred_class))